# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [87]:
#1 Imports and Warnings Suppression

In [103]:
import requests
from bs4 import BeautifulSoup
from click import prompt
from itext2kg import iText2KG_Star,DocumentsDistiller
from newspaper import Article
from youtube_transcript_api import YouTubeTranscriptApi
import yaml
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.caches import BaseCache
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator
from langchain_core.callbacks.base import Callbacks
from typing import List
from iText2KG.itext2kg import iText2KG
import networkx as nx
import matplotlib as plt
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from neo4j_client import Neo4jClient
import traceback
import json
import os
import re
import asyncio
from enum import Enum
from itext2kg.models.schemas import Facts
from typing import List, Dict, Any, Optional
from itext2kg import iText2KG_Star
from itext2kg.logging_config import setup_logging, get_logger
from langchain_core.caches import BaseCache
from typing import List, Tuple, Set
from collections import defaultdict, Counter

# Suppress Pydantic schema warning
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()
ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()


In [104]:
#2 Article Scraping Function 

In [105]:
def scrape_article(url):
    print(f"Scraping article from: {url}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        # Method 1: BeautifulSoup approach
        soup = BeautifulSoup(response.text, 'html.parser')
        content_div = soup.find("div", class_="entry-content ap-pattern--entry-content")

        if content_div:
            article_text = content_div.get_text(separator="\n", strip=True)
            with open("output/pure_scrum_clean.txt", "w", encoding="utf-8") as f:
                f.write(article_text)
            print("Article scraped successfully with BeautifulSoup")
        else:
            print("Specific div not found, trying newspaper3k...")

        # Method 2: Newspaper3k approach (as backup)
        article = Article(url)
        article.download()
        article.parse()

        if article.text:
            with open("output/pure_scrum_clean2.txt", "w", encoding="utf-8") as f:
                f.write(article.text)
            print("Article scraped successfully with newspaper3k")
            return article.text
        else:
            print("Failed to extract article content")
            return None

    except requests.RequestException as e:
        print(f"Error fetching article: {e}")
        return None

In [106]:
#3. YouTube Transcript Retrieval Function

In [107]:
def get_youtube_transcript(video_id):
    print(f"Getting YouTube transcript for video: {video_id}")

    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        transcript = " ".join([item['text'] for item in transcript_list])

        with open("output/youtubeVideo.txt", "w", encoding="utf-8") as f:
            f.write(transcript)

        print("YouTube transcript downloaded successfully")
        return transcript

    except Exception as e:
        print(f"Error getting YouTube transcript: {e}")
        return None

In [108]:
#4. Configuration Loading

In [109]:
def load_config(config_path="config.yml"):
    try:
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        print("config.yml not found. Creating sample config for Ollama...")

        sample_config = {
            'llm': {
                'model_type': 'ollama',
                'model_name': 'llama3.1',
                'format': 'json',
                'temperature': 0.1,
                'max_tokens': 2000,
                'timeout': 60,
                'max_retries': 3
            },
            'paths': {
                'output_dir': './output'
            }
        }

        with open(config_path, "w") as f:
            yaml.dump(sample_config, f, default_flow_style=False)

        print("Sample config.yml created. Update it if needed.")
        return sample_config


In [110]:
#5. LLM and Embeddings Initialization Open AI

In [111]:
# import os
# 
# 
# def initialize_llm(config):
#     if not config:
#         return None, None
#     
#     llm_config = config.get('llm', {})
#     api_key = llm_config.get('api_key')
#     
#     if not api_key:
#         api_key = os.getenv('OPENAI_API_KEY')
#         if not api_key:
#             print("OpenAI API key not found. Please set OPENAI_API_KEY environment variable or update config.yml")
#             return None, None
#     
#     try:
#         if llm_config.get('model_type') == "openai":
#             llm = ChatOpenAI(
#                 model=llm_config.get('model_name', 'gpt-3.5-turbo'),
#                 temperature=llm_config.get('temperature', 0.7),
#                 max_tokens=llm_config.get('max_tokens', 2000),
#                 api_key=api_key,
#                 request_timeout=llm_config.get('timeout', 60),
#                 max_retries=llm_config.get('max_retries', 3)
#             )
#             embeddings = OpenAIEmbeddings(api_key=api_key)
#             print("LLM and Embeddings initialized successfully")
#             return llm, embeddings
#         else:
#             print("Only 'openai' model type is supported")
#             return None, None
#     except Exception as e:
#         print(f"Error initializing LLM: {e}")
#         return None, None
#     

In [112]:
#5.1 Ollama

In [113]:

def initialize_llm(config):
    if not config:
        return None, None

    llm_config = config.get('llm', {})

    try:
        if llm_config.get('model_type') == "ollama":
            llm = ChatOllama(
                model=llm_config.get('model_name', 'llama3.1'),
                temperature=llm_config.get('temperature', 0.1)
            )

            embeddings = OllamaEmbeddings(model=llm_config.get('model_name', 'llama3.1'))

            print("LLM and Embeddings initialized successfully (Ollama)")
            return llm, embeddings
        else:
            print("Only 'ollama' model type is supported in this setup")
            return None, None
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None, None


In [114]:
#8. Save Triples to File and also filter duplicates

In [115]:
# # def normalize_triples(triples):
# #     return [(s.strip().lower(), p.strip().lower(), o.strip().lower()) for s, p, o in triples]
# # 
# # 
# # def filter_triples(triples):
# #     filtered = []
# #     for subj, pred, obj in triples:
# #         if subj and pred and obj and subj.lower() != 'none' and obj.lower() != 'none':
# #             filtered.append((subj, pred, obj))
# #     return filtered
# # 
# # 
# # def remove_duplicates(triples):
# #     seen = set()
# #     unique_triples = []
# #     for triple in triples:
# #         if triple not in seen:
# #             unique_triples.append(triple)
# #             seen.add(triple)
# #     return unique_triples
# # 
# # 
def save_triples(triples, filename="extracted_triples.txt"):
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write("Extracted Knowledge Graph Triples:\n")
            f.write("=" * 40 + "\n\n")
            for i, (subject, predicate, obj) in enumerate(triples, 1):
                f.write(f"{i}. ({subject}) --[{predicate}]--> ({obj})\n")
        print(f"Triples saved to {filename}")
    except Exception as e:
        print(f"Error saving triples: {e}")

# 
# def normalize_triples(triples):
#     normalized = []
#     for triple in triples:
#         if len(triple) == 3:
#             subject, predicate, obj = triple
#             subject = str(subject).strip().lower()
#             predicate = str(predicate).strip().lower().replace('_', ' ')
#             obj = str(obj).strip().lower()
#             normalized.append((subject, predicate, obj))
#     return normalized
# 
# 
# def filter_triples(triples):
#     filtered = []
#     for triple in triples:
#         if len(triple) == 3:
#             subject, predicate, obj = triple
#             if len(subject) > 1 and len(predicate) > 1 and len(obj) > 1:
#                 if subject != obj:
#                     filtered.append(triple)
#     return filtered
# 
# 
# def remove_duplicates(triples):
#     return list(set(triples))
# 
# def semantic_filter(triples):
#     hallucinated_terms = {"family", "romantic", "divorce"}
#     return [
#         t for t in triples
#         if not any(term in str(t).lower() for term in hallucinated_terms)
#     ]
# 
# # def save_triples(triples, filename="extracted_triples.txt"):
# #     with open(filename, 'w', encoding='utf-8') as f:
# #         for subject, predicate, obj in triples:
# #             f.write(f"{subject}\t{predicate}\t{obj}\n")


In [116]:
#11. Plots

In [117]:
import networkx as nx
import matplotlib.pyplot as plt


def plot_triples(triples):
    G = nx.DiGraph()
    for subj, pred, obj in triples:
        G.add_node(subj)
        G.add_node(obj)
        G.add_edge(subj, obj, label=pred)

    pos = nx.spring_layout(G, k=0.5, iterations=50)

    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=10, arrowsize=20)

    edge_labels = {(subj, obj): pred for subj, pred, obj in triples}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)

    plt.title("Knowledge Graph Visualization")
    plt.show()


In [118]:
#9. Main Execution Logic

In [119]:
# import re
# def main():
#     print("Starting Knowledge Graph Extraction Pipeline\n")
# 
#     # Scrape article
#     # url = "https://age-of-product.com/pure-scrum/"
#     # article_text = scrape_article(url)
#     # 
#     # # Get YouTube transcript
#     # video_id = "502ILHjX9EE"
#     # youtube_text = get_youtube_transcript(video_id)
# 
#     # Choose which text to process
#     # if youtube_text:
#     #     input_text = youtube_text
#     #     print("Using YouTube transcript for KG extraction")
#     # elif article_text:
#     #     input_text = article_text
#     #     print("Using article text for KG extraction")
#     # else:
#     #     print("No text available for processing")
#     #     return
# 
#     try:
#         with open("text.txt", "r", encoding="utf-8") as f:
#             input_text = f.read()
#     except Exception as e:
#         print(f"Error reading 'text.txt': {e}")
#         return
# 
#     config = load_config()
#     llm, embeddings = initialize_llm(config)
# 
#     if not llm:
#         print("Cannot proceed without LLM initialization")
#         return
# 
#     print(f"\nInput text length: {len(input_text)} characters")
#     # all_triples = process_text_in_chunks(input_text, llm,delay_between_chunks=0)
# 
#     # shall try this next automation
#     kg = iText2KG(llm,embeddings)
# 
#     doc = {
#         "text": input_text,
#         "title": "ManualText",
#         "meta": {
#             "source": "manual_input"
#         }
#     }
#     try:
#         response = llm.invoke(f"Extract triples from text:\n\n{input_text[:1000]}")
#         print("Raw LLM response:\n", response.content)
# 
#     except Exception as e:
#         print(f"Extraction failed: {e}")
#         return
# 
#     def parse_llm_response_to_triples(text):
#         triples = []
#         pattern = re.compile(r"^\s*\d+\.\s*(.+?)\s*-\s*(.+)$")
#         for line in text.splitlines():
#             match = pattern.match(line)
#             if match:
#                 subj = match.group(1).strip()
#                 rest = match.group(2).strip()
#                 parts = rest.split(' - ')
#                 if len(parts) == 2:
#                     pred = parts[0].strip()
#                     obj = parts[1].strip()
#                 else:
#                     pred = rest
#                     obj = ""
#                 triples.append((subj, pred, obj))
#         return triples
# 
#     
#     all_triples = parse_llm_response_to_triples(response.content)
# 
#     if all_triples:
#         print(f"\nTotal triples extracted: {len(all_triples)}")
#         all_triples = normalize_triples(all_triples)
#         all_triples = filter_triples(all_triples)
#         all_triples = remove_duplicates(all_triples)
#         save_triples(all_triples)
# 
#         print(Neo4jClient.__init__)
#         neo4j_client = Neo4jClient(uri="bolt://localhost:7687", user="neo4j", password="12345678")
#         neo4j_client.create_triples(all_triples)
#         neo4j_client.close()
# 
#         print("\nSample triples:")
#         for i, (subject, predicate, obj) in enumerate(all_triples[:5], 1):
#             print(f"  {i}. ({subject}) --[{predicate}]--> ({obj})")
# 
#         if len(all_triples) > 5:
#             print(f"  ... and {len(all_triples) - 5} more triples")
# 
#         plot_triples(all_triples)
#     else:
#         print("No triples were successfully extracted")




In [120]:
#NOT USING STAR(TRADITIONAL)
async def main_traditional_itext2kg(input_source: str = "text.txt"):
    print("Starting Knowledge Graph Extraction with Traditional iText2KG\n")

    # ============ LOAD INPUT TEXTS ============
    texts = []
    try:
        if isinstance(input_source, str) and os.path.exists(input_source):
            print(f"Loading from file: {input_source}")
            with open(input_source, "r", encoding="utf-8") as f:
                content = f.read().strip()

                if len(content) > 2000:
                    sentences = content.split('. ')
                    current_chunk = ""
                    for sentence in sentences:
                        if len(current_chunk + sentence) <= 2000:
                            current_chunk += sentence + ". "
                        else:
                            if current_chunk:
                                texts.append(current_chunk.strip())
                            current_chunk = sentence + ". "
                    if current_chunk:
                        texts.append(current_chunk.strip())
                else:
                    texts = [content]

        elif isinstance(input_source, str) and len(input_source.strip()) > 10:
            print("Using direct text input")
            texts = [input_source.strip()]

        else:
            print(f"Invalid input source: {input_source}")
            return []

    except Exception as e:
        print(f"Error loading input: {e}")
        return []

    texts = [text for text in texts if len(text.strip()) > 20]

    if not texts:
        print("No valid texts found. Exiting.")
        return []

    print(f"Loaded {len(texts)} text segments")
    for i, text in enumerate(texts[:2]):
        print(f"   Text {i + 1} preview: {text[:100]}...")

    # ============ INITIALIZE LLM ============
    print("\nInitializing LLM and Embeddings...")
    config = load_config()
    llm, embeddings = initialize_llm(config)

    if not llm or not embeddings:
        print("Cannot proceed without LLM initialization")
        return []

    print(f"LLM initialized: {config['llm']['model_name']}")

    # ============ KNOWLEDGE GRAPH CONSTRUCTION WITH TRADITIONAL iText2KG ============
    print("\nBuilding Knowledge Graph with Traditional iText2KG...")

    all_triples = []

    try:
        from itext2kg import iText2KG

        itext2kg = iText2KG(llm_model=llm, embeddings_model=embeddings)
        print("Traditional iText2KG initialized successfully")

        print(f"Building knowledge graph from {len(texts)} sections...")

        kg_result = await itext2kg.build_graph(
            sections=texts,
            ent_threshold=0.7,
            rel_threshold=0.7,
            max_tries=3
        )

        print(f"Knowledge graph construction completed")
        print(f"Result type: {type(kg_result)}")

        # ============ EXTRACT TRIPLES FROM RESULT ============
        if hasattr(kg_result, 'relationships'):
            relationships = kg_result.relationships
            print(f"Found {len(relationships)} relationships")
        elif hasattr(kg_result, 'edges'):
            relationships = kg_result.edges
            print(f"Found {len(relationships)} edges")
        elif isinstance(kg_result, dict) and 'relationships' in kg_result:
            relationships = kg_result['relationships']
            print(f"Found {len(relationships)} relationships in dict")
        elif isinstance(kg_result, list):
            relationships = kg_result
            print(f"Found {len(relationships)} items in list")
        else:
            print("Unknown result format from iText2KG")
            print(f"Available attributes: {dir(kg_result) if hasattr(kg_result, '__dict__') else 'Not an object'}")
            relationships = []

        for i, rel in enumerate(relationships):
            try:
                if hasattr(rel, '__class__') and 'Relationship' in str(rel.__class__):
                    print(f"Processing Relationship object {i}")

                    subject = None
                    predicate = None
                    obj = None

                    if hasattr(rel, 'name'):
                        predicate = str(rel.name).strip()

                    if hasattr(rel, 'startEntity'):
                        start_entity = rel.startEntity
                        if hasattr(start_entity, 'label'):
                            subject = str(start_entity.label).strip()
                        elif hasattr(start_entity, 'name'):
                            subject = str(start_entity.name).strip()

                    if hasattr(rel, 'endEntity'):
                        end_entity = rel.endEntity
                        if hasattr(end_entity, 'label'):
                            obj = str(end_entity.label).strip()
                        elif hasattr(end_entity, 'name'):
                            obj = str(end_entity.name).strip()

                    print(f"  Extracted - Subject: {subject}, Predicate: {predicate}, Object: {obj}")

                elif isinstance(rel, dict):
                    subject = rel.get('subject') or rel.get('source') or rel.get('head')
                    predicate = rel.get('predicate') or rel.get('relation') or rel.get('type')
                    obj = rel.get('object') or rel.get('target') or rel.get('tail')

                elif isinstance(rel, (tuple, list)) and len(rel) >= 3:
                    subject, predicate, obj = rel[0], rel[1], rel[2]

                else:
                    print(f"Unknown relationship format at index {i}: {type(rel)}")
                    continue

                if subject and predicate and obj:
                    subject = str(subject).strip()
                    predicate = str(predicate).strip()
                    obj = str(obj).strip()

                    if len(subject) > 1 and len(obj) > 1 and subject != obj and len(predicate) > 1:
                        all_triples.append((subject, predicate, obj))
                        print(f"Added triple: ({subject}) --[{predicate}]--> ({obj})")
                    else:
                        print(f"Skipped invalid triple: subject='{subject}', predicate='{predicate}', object='{obj}'")
                else:
                    print(f"Could not extract complete triple from relationship {i}")
                    print(f"  Subject: {subject}, Predicate: {predicate}, Object: {obj}")

            except Exception as e:
                print(f"Error processing relationship {i}: {e}")
                import traceback
                traceback.print_exc()
                continue

    except Exception as e:
        print(f"Knowledge graph construction failed: {e}")
        import traceback
        traceback.print_exc()
        return []

    # ============ POST-PROCESSING & OUTPUT ============
    print(f"\nProcessing Results...")

    if all_triples:
        original_count = len(all_triples)
        all_triples = list(set(all_triples))
        print(f"After deduplication: {len(all_triples)} triples (removed {original_count - len(all_triples)})")

        save_triples(all_triples, "extracted_triples.txt")

        print("\nEXTRACTED KNOWLEDGE GRAPH:")
        print("=" * 50)
        for i, (subject, predicate, obj) in enumerate(all_triples[:10], 1):
            print(f"  {i:2d}. ({subject}) --[{predicate}]--> ({obj})")

        if len(all_triples) > 10:
            print(f"  ... and {len(all_triples) - 10} more triples")

        print(f"\nSuccess! Extracted {len(all_triples)} knowledge relationships")

        try:
            plot_triples(all_triples[:20])
        except Exception as e:
            print(f"Visualization skipped: {e}")

        return all_triples

    else:
        print("\nNo triples were successfully extracted")
        return []

In [121]:
# # ================= ENUMS =================
# class ScrumEntityType(str, Enum):
#     PRODUCT_OWNER = "product_owner"
#     SCRUM_MASTER = "scrum_master"
#     DEVELOPER = "developer"
#     SCRUM_TEAM = "scrum_team"
#     STAKEHOLDER = "stakeholder"
#     PRODUCT_BACKLOG = "product_backlog"
#     SPRINT_BACKLOG = "sprint_backlog"
#     INCREMENT = "increment"
#     PRODUCT_GOAL = "product_goal"
#     SPRINT_GOAL = "sprint_goal"
#     SPRINT = "sprint"
#     SPRINT_PLANNING = "sprint_planning"
#     DAILY_SCRUM = "daily_scrum"
#     SPRINT_REVIEW = "sprint_review"
#     SPRINT_RETROSPECTIVE = "sprint_retrospective"
#     USER_STORY = "user_story"
#     FEATURE = "feature"
#     BUG = "bug"
#     TASK = "task"
#     TECHNICAL_DEBT = "technical_debt"
#     UNKNOWN = "unknown"  # fallback
# 
# class ScrumRelationType(str, Enum):
#     MANAGES = "manages"
#     OWNS = "owns"
#     DEVELOPS = "develops"
#     FACILITATES = "facilitates"
#     CREATES = "creates"
#     PARTICIPATES_IN = "participates_in"
#     RELATES_TO = "relates_to"  # fallback
# 
# # ================= FACTS =================
# class ScrumFact(BaseModel):
#     statement: str = Field(default="")
#     subject: str
#     predicate: str
#     object: str = Field(default="unknown") 
#     subject_type: ScrumEntityType = ScrumEntityType.UNKNOWN
#     object_type: ScrumEntityType = ScrumEntityType.UNKNOWN
#     relation_type: ScrumRelationType = ScrumRelationType.RELATES_TO
#     confidence: float = 0.8
#     scrum_context: str = "General"
#     
#     
#     # ================= VALIDATORS =================
#     
#     @model_validator(mode="before")
#     def ensure_object(cls, values):
#         if "object" not in values or not values["object"]:
#             values["object"] = "unknown"
#         return values
#     
#     @field_validator("subject_type", "object_type", mode="before")
#     @classmethod
#     def normalize_entity(cls, v):
#         if isinstance(v, str):
#             return ENTITY_MAP.get(v.lower().strip(), ScrumEntityType.UNKNOWN)
#         return v
# 
#     @field_validator("relation_type", mode="before")
#     @classmethod
#     def normalize_relation(cls, v):
#         if isinstance(v, str):
#             return RELATION_MAP.get(v.lower().strip(), ScrumRelationType.RELATES_TO)
#         return v
# 
# # ================= PROJECT FACTS =================
# class ScrumProjectFacts(BaseModel):
#     facts: List[ScrumFact] = []
# 
# # ================= MAPPING DICTS =================
# ENTITY_MAP = {
#     # Roles
#     "po": ScrumEntityType.PRODUCT_OWNER,
#     "product owner": ScrumEntityType.PRODUCT_OWNER,
#     "scrum master": ScrumEntityType.SCRUM_MASTER,
#     "sm": ScrumEntityType.SCRUM_MASTER,
#     "developer": ScrumEntityType.DEVELOPER,
#     "dev": ScrumEntityType.DEVELOPER,
#     "team": ScrumEntityType.SCRUM_TEAM,
#     "scrum team": ScrumEntityType.SCRUM_TEAM,
#     "dev team": ScrumEntityType.SCRUM_TEAM,
#     "stakeholder": ScrumEntityType.STAKEHOLDER,
#     
#     # Artifacts
#     "product backlog": ScrumEntityType.PRODUCT_BACKLOG,
#     "backlog": ScrumEntityType.PRODUCT_BACKLOG,
#     "sprint backlog": ScrumEntityType.SPRINT_BACKLOG,
#     "increment": ScrumEntityType.INCREMENT,
#     "product goal": ScrumEntityType.PRODUCT_GOAL,
#     "sprint goal": ScrumEntityType.SPRINT_GOAL,
#     "sprint": ScrumEntityType.SPRINT,
#     "user story": ScrumEntityType.USER_STORY,
#     "feature": ScrumEntityType.FEATURE,
#     "bug": ScrumEntityType.BUG,
#     "task": ScrumEntityType.TASK,
#     "technical debt": ScrumEntityType.TECHNICAL_DEBT,
#     
#     # Events
#     "sprint planning": ScrumEntityType.SPRINT_PLANNING,
#     "daily scrum": ScrumEntityType.DAILY_SCRUM,
#     "sprint review": ScrumEntityType.SPRINT_REVIEW,
#     "sprint retrospective": ScrumEntityType.SPRINT_RETROSPECTIVE,
# }
# RELATION_MAP = {
#     "manages": ScrumRelationType.MANAGES,
#     "responsible for": ScrumRelationType.MANAGES,
#     "facilitates": ScrumRelationType.FACILITATES,
#     "owns": ScrumRelationType.OWNS,
#     "creates": ScrumRelationType.CREATES,
#     "develops": ScrumRelationType.DEVELOPS,
#     "participates in": ScrumRelationType.PARTICIPATES_IN,
#     "attends": ScrumRelationType.PARTICIPATES_IN,
#     "works on": ScrumRelationType.DEVELOPS,
#     "related to": ScrumRelationType.RELATES_TO,
# }
# 
# 
# # ============ SIMPLIFIED IE QUERY ============
# def get_scrum_IE_query():
#     return '''
#     # You are a Scrum expert analyzing project documentation.
#     # Extract facts about WHO does WHAT in the Scrum context.
#     
#     OUTPUT STRICTLY IN THIS JSON FORMAT as a list of facts:
#     [
#       {
#         "subject": "<person/role/entity name>",
#         "predicate": "<action/relationship>", 
#         "object": "<target entity name>",
#         "subject_type": "<entity_type>",
#         "object_type": "<entity_type>",
#         "relation_type": "<relation_type>",
#         "confidence": <0.0-1.0>,
#         "scrum_context": "<context description>",
#         "statement": "<full descriptive statement>"
#       }
#     ]
#     
#     # Entity Types (use these exactly):
#     - product_owner, scrum_master, developer, scrum_team, stakeholder
#     - product_backlog, sprint_backlog, increment, product_goal, sprint_goal
#     - sprint, sprint_planning, daily_scrum, sprint_review, sprint_retrospective  
#     - user_story, feature, bug, task, technical_debt
#     - unknown (if unsure)
#     
#     # Relation Types (use these exactly):
#     - manages, owns, develops, facilitates, creates, participates_in, relates_to
#     
#     # Examples:
#     [
#       {
#         "subject": "Pat",
#         "predicate": "manages",
#         "object": "Product Backlog", 
#         "subject_type": "product_owner",
#         "object_type": "product_backlog",
#         "relation_type": "manages",
#         "confidence": 0.9,
#         "scrum_context": "Product ownership responsibilities",
#         "statement": "Pat manages the Product Backlog"
#       },
#       {
#         "subject": "Scrum Master",
#         "predicate": "facilitates", 
#         "object": "Sprint Planning",
#         "subject_type": "scrum_master",
#         "object_type": "sprint_planning", 
#         "relation_type": "facilitates",
#         "confidence": 0.95,
#         "scrum_context": "Sprint event facilitation",
#         "statement": "Scrum Master facilitates Sprint Planning"
#       }
#     ]
#     
#     # Rules:
#     - Return ONLY a JSON list, no additional text
#     - Use exact entity and relation types from the lists above
#     - Set confidence between 0.7-1.0 based on clarity
#     - Make statements descriptive but concise
#     - Focus on Scrum roles, artifacts, events, and their relationships
#     '''

class ScrumFact(BaseModel):
    subject: str
    predicate: str
    object: str
    confidence: float = 0.8


class ScrumProjectFacts(BaseModel):
    facts: List[ScrumFact] = []

    @field_validator("facts", mode="before")
    @classmethod
    def parse_json_string(cls, v):
        if isinstance(v, str):
            try:
                parsed = json.loads(v)
                if isinstance(parsed, list):
                    return [ScrumFact(**item) for item in parsed]
            except:
                pass
        return v

# ============ SIMPLE IE QUERY ============
def get_scrum_IE_query():
    return '''Extract Scrum relationships as JSON:

[
  {"subject": "who", "predicate": "does what", "object": "to what", "confidence": 0.8}
]

Examples:
[
  {"subject": "Pat", "predicate": "manages", "object": "Product Backlog", "confidence": 0.9},
  {"subject": "Scrum Master", "predicate": "facilitates", "object": "Sprint Planning", "confidence": 0.8},
  {"subject": "Developer", "predicate": "works on", "object": "User Story", "confidence": 0.7}
]

Return only JSON array, no text.'''

In [122]:
# async def main(input_source: str = "text.txt"):
#     print("Starting Enhanced Knowledge Graph Extraction with Scrum Schema + iText2KG_Star\n")
# 
#     # ============ LOAD INPUT TEXTS ============
#     texts = []
#     try:
#         if isinstance(input_source, str) and os.path.exists(input_source):
#             print(f"Loading from file: {input_source}")
#             with open(input_source, "r", encoding="utf-8") as f:
#                 content = f.read().strip()
#     
#                 if len(content) > 2000:
#                     sentences = content.split('. ')
#                     current_chunk = ""
#                     for sentence in sentences:
#                         if len(current_chunk + sentence) <= 2000:
#                             current_chunk += sentence + ". "
#                         else:
#                             if current_chunk:
#                                 texts.append(current_chunk.strip())
#                             current_chunk = sentence + ". "
#                     if current_chunk:
#                         texts.append(current_chunk.strip())
#                 else:
#                     texts = [content]
#     
#         elif isinstance(input_source, str) and len(input_source.strip()) > 10:
#             print("Using direct text input")
#             texts = [input_source.strip()]
#     
#         else:
#             print(f"Invalid input source: {input_source}")
#             return []
#     
#     except Exception as e:
#         print(f"Error loading input: {e}")
#         return []
#     
#     texts = [text for text in texts if len(text.strip()) > 20]
#     
#     if not texts:
#         print("No valid texts found. Exiting.")
#         return []
#     
#     print(f"Loaded {len(texts)} text segments")
#     for i, text in enumerate(texts[:2]):
#         print(f"   Text {i + 1} preview: {text[:100]}...")
#     
#     # ============ INITIALIZE LLM ============
#     print("\nInitializing LLM and Embeddings...")
#     config = load_config()
#     llm, embeddings = initialize_llm(config)
#     
#     if not llm or not embeddings:
#         print("Cannot proceed without LLM initialization")
#         return []
#     
#     print(f"LLM and Embeddings initialized successfully (Ollama)")
#     print(f"LLM initialized: {config['llm']['model_name']}")
# 
#     # ============ COMPREHENSIVE EXTRACTION (ALL STRATEGIES IN ONE) ============
#     print("\nExtracting knowledge with multiple methods...")
#     
#     all_triples = []
#     
#     # METHOD 1: Try iText2KG_Star first (but don't rely on it)
#     print("Trying iText2KG_Star...")
#     try:
#         from itext2kg import iText2KG_Star
#         itext2kg_star = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)
#         
#         kg_result = await itext2kg_star.build_graph(
#             sections=texts,
#             ent_threshold=0.6,  # More forgiving
#             rel_threshold=0.6,
#             max_tries=2,
#             entity_name_weight=0.6,
#             entity_label_weight=0.4,
#             observation_date="2025-08-10"
#         )
#         
#         if kg_result and hasattr(kg_result, 'relationships'):
#             for rel in kg_result.relationships:
#                 try:
#                     # Extract subject
#                     subject = None
#                     if hasattr(rel, 'startEntity'):
#                         if hasattr(rel.startEntity, 'name') and rel.startEntity.name:
#                             subject = str(rel.startEntity.name).strip().title()
#                         elif hasattr(rel.startEntity, 'label') and rel.startEntity.label:
#                             subject = str(rel.startEntity.label).strip().title()
#                     
#                     # Extract object
#                     obj = None
#                     if hasattr(rel, 'endEntity'):
#                         if hasattr(rel.endEntity, 'name') and rel.endEntity.name:
#                             obj = str(rel.endEntity.name).strip().title()
#                         elif hasattr(rel.endEntity, 'label') and rel.endEntity.label:
#                             obj = str(rel.endEntity.label).strip().title()
#                     
#                     # Extract predicate
#                     predicate = "relates_to"
#                     if hasattr(rel, 'name') and rel.name:
#                         predicate = str(rel.name).strip().lower().replace(' ', '_')
#                     
#                     if subject and obj and subject != obj and len(subject) > 1 and len(obj) > 1:
#                         all_triples.append((subject, predicate, obj))
#                 except:
#                     continue
#             
#             print(f"✓ iText2KG_Star found {len([t for t in all_triples])} triples")
#         else:
#             print("✗ iText2KG_Star returned no results")
#             
#     except Exception as e:
#         print(f"✗ iText2KG_Star failed: {e}")
#     
#     # METHOD 2: Document Distiller (if available)
#     print("Trying Document Distiller...")
#     try:
#         from itext2kg import DocumentsDistiller
#         
#         document_distiller = DocumentsDistiller(llm_model=llm)
#         scrum_IE_query = get_scrum_IE_query()
#         
#         for i, text in enumerate(texts):
#             try:
#                 facts_result = await document_distiller.distill(
#                     documents=[text],
#                     IE_query=scrum_IE_query,
#                     output_data_structure=ScrumProjectFacts
#                 )
#                 
#                 if facts_result and hasattr(facts_result, 'facts'):
#                     for fact in facts_result.facts:
#                         if (hasattr(fact, 'subject') and hasattr(fact, 'object') and 
#                             fact.subject and fact.object):
#                             
#                             subject = str(fact.subject).strip().title()
#                             obj = str(fact.object).strip().title()
#                             
#                             # Get predicate - try multiple sources
#                             predicate = "relates_to"
#                             if hasattr(fact, 'predicate') and fact.predicate:
#                                 predicate = str(fact.predicate).strip().lower()
#                             elif hasattr(fact, 'relation_type') and fact.relation_type:
#                                 predicate = str(fact.relation_type).strip().lower()
#                             
#                             if subject and obj and subject != obj and len(subject) > 1 and len(obj) > 1:
#                                 all_triples.append((subject, predicate, obj))
#                 
#             except Exception as e:
#                 continue
#         
#         distiller_count = len(all_triples) - len([t for t in all_triples if t not in all_triples[:len(all_triples)//2]])
#         print(f"✓ Document Distiller added triples")
#         
#     except Exception as e:
#         print(f"✗ Document Distiller failed: {e}")
#     
#     # METHOD 3: COMPREHENSIVE PATTERN MATCHING (The Real Fix)
#     print("Applying comprehensive pattern matching...")
#     
#     # Expanded patterns - covers way more relationships
#     patterns = [
#         # Basic role patterns
#         (r'(\w+(?:\s+\w+)*)\s+is\s+(?:a\s+|an\s+|the\s+)?([A-Z][^.]*?)(?:\.|$)', 'is'),
#         (r'(\w+(?:\s+\w+)*)\s+acts\s+as\s+(?:a\s+|an\s+|the\s+)?([^.]*?)(?:\.|$)', 'acts_as'),
#         
#         # Management and ownership
#         (r'(\w+(?:\s+\w+)*)\s+(?:is\s+)?responsible\s+for\s+([^.]*?)(?:\.|$)', 'responsible_for'),
#         (r'(\w+(?:\s+\w+)*)\s+(?:is\s+)?accountable\s+for\s+([^.]*?)(?:\.|$)', 'accountable_for'),
#         (r'(\w+(?:\s+\w+)*)\s+manages?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'manages'),
#         (r'(\w+(?:\s+\w+)*)\s+owns?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'owns'),
#         (r'(\w+(?:\s+\w+)*)\s+oversees?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'oversees'),
#         (r'(\w+(?:\s+\w+)*)\s+maintains?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'maintains'),
#         (r'(\w+(?:\s+\w+)*)\s+leads?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'leads'),
#         
#         # Actions and creation
#         (r'(\w+(?:\s+\w+)*)\s+creates?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'creates'),
#         (r'(\w+(?:\s+\w+)*)\s+develops?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'develops'),
#         (r'(\w+(?:\s+\w+)*)\s+builds?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'builds'),
#         (r'(\w+(?:\s+\w+)*)\s+produces?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'produces'),
#         (r'(\w+(?:\s+\w+)*)\s+delivers?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'delivers'),
#         (r'(\w+(?:\s+\w+)*)\s+provides?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'provides'),
#         (r'(\w+(?:\s+\w+)*)\s+ensures?\s+(?:that\s+)?([^.]*?)(?:\.|$)', 'ensures'),
#         
#         # Collaboration  
#         (r'(\w+(?:\s+\w+)*)\s+works?\s+with\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'works_with'),
#         (r'(\w+(?:\s+\w+)*)\s+collaborates?\s+with\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'collaborates_with'),
#         (r'(\w+(?:\s+\w+)*)\s+communicates?\s+with\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'communicates_with'),
#         (r'(\w+(?:\s+\w+)*)\s+coordinates?\s+(?:with\s+)?(?:the\s+)?([^.]*?)(?:\.|$)', 'coordinates'),
#         (r'(\w+(?:\s+\w+)*)\s+facilitates?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'facilitates'),
#         (r'(\w+(?:\s+\w+)*)\s+supports?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'supports'),
#         (r'(\w+(?:\s+\w+)*)\s+helps?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'helps'),
#         
#         # Events and activities
#         (r'(\w+(?:\s+\w+)*)\s+participates?\s+in\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'participates_in'),
#         (r'(\w+(?:\s+\w+)*)\s+attends?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'attends'),
#         (r'(\w+(?:\s+\w+)*)\s+conducts?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'conducts'),
#         (r'(\w+(?:\s+\w+)*)\s+runs?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'runs'),
#         (r'(\w+(?:\s+\w+)*)\s+hosts?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'hosts'),
#         (r'(\w+(?:\s+\w+)*)\s+organizes?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'organizes'),
#         
#         # Review and quality
#         (r'(\w+(?:\s+\w+)*)\s+reviews?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'reviews'),
#         (r'(\w+(?:\s+\w+)*)\s+inspects?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'inspects'),
#         (r'(\w+(?:\s+\w+)*)\s+evaluates?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'evaluates'),
#         (r'(\w+(?:\s+\w+)*)\s+defines?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'defines'),
#         (r'(\w+(?:\s+\w+)*)\s+specifies?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'specifies'),
#         (r'(\w+(?:\s+\w+)*)\s+prioritizes?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'prioritizes'),
#         (r'(\w+(?:\s+\w+)*)\s+refines?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'refines'),
#         
#         # Artifacts and containers
#         (r'(\w+(?:\s+\w+)*)\s+contains?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'contains'),
#         (r'(\w+(?:\s+\w+)*)\s+includes?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'includes'),
#         (r'(\w+(?:\s+\w+)*)\s+consists?\s+of\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'consists_of'),
#         (r'(\w+(?:\s+\w+)*)\s+has\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'has'),
#         (r'(\w+(?:\s+\w+)*)\s+comprises?\s+(?:the\s+)?([^.]*?)(?:\.|$)', 'comprises'),
#     ]
#     
#     # Apply patterns to all text
#     pattern_triples = []
#     for text in texts:
#         # Split into sentences for better matching
#         sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 5]
#         
#         for sentence in sentences:
#             for pattern, relation in patterns:
#                 matches = re.finditer(pattern, sentence, re.IGNORECASE)
#                 
#                 for match in matches:
#                     subject = match.group(1).strip().title()
#                     obj = match.group(2).strip()
#                     
#                     # Clean up the object
#                     obj = re.sub(r'\s+', ' ', obj).strip()  # Remove extra spaces
#                     obj = obj.replace(',', '').replace(';', '')  # Remove punctuation
#                     if obj.lower().startswith('the '):
#                         obj = obj[4:]
#                     
#                     # Limit object length and clean up
#                     if len(obj) > 50:  # Skip overly long objects
#                         continue
#                     
#                     # Clean common verbose patterns
#                     obj = obj.replace(' And ', ' and ')
#                     obj = obj.replace(' Or ', ' or ')
#                     obj = obj.replace(' The ', ' the ')
#                     obj = obj.title()
#                     
#                     # Filter valid triples
#                     if (len(subject) > 1 and len(obj) > 1 and len(obj) < 50 and  # Shorter length limit
#                         subject.lower() != obj.lower() and  # Not self-referential
#                         not obj.lower() in ['it', 'this', 'that', 'they', 'them'] and  # No pronouns
#                         len([c for c in subject if c.isalpha()]) > 1 and  # Has actual letters
#                         not any(word in obj.lower() for word in ['maybe', 'somebody', 'here are']) and  # Remove vague terms
#                         obj.count(' ') < 8):  # Not too many words
#                         
#                         pattern_triples.append((subject, relation, obj))
#     
#     all_triples.extend(pattern_triples)
#     print(f"✓ Pattern matching found {len(pattern_triples)} triples")
#     
#     # METHOD 4: COREFERENCE RESOLUTION
#     print("Applying coreference resolution...")
#     
#     # Build entity mapping (Pat -> Product Owner, etc.)
#     entity_map = {}
#     role_indicators = ['is a', 'is the', 'acts as', 'serves as', 'works as']
#     
#     for text in texts:
#         # Look for role assignments
#         for indicator in role_indicators:
#             pattern = rf'(\w+(?:\s+\w+)*)\s+{re.escape(indicator)}\s+([A-Z][^.]*?)(?:\.|$)'
#             matches = re.finditer(pattern, text, re.IGNORECASE)
#             
#             for match in matches:
#                 person = match.group(1).strip().title()
#                 role = match.group(2).strip().title()
#                 
#                 if len(person) > 1 and len(role) > 1 and person != role:
#                     entity_map[person] = role
#                     print(f"Mapped: {person} -> {role}")
#     
#     # Apply coreference resolution to triples
#     resolved_triples = []
#     for subject, predicate, obj in all_triples:
#         # Resolve subject
#         if subject in entity_map:
#             resolved_subject = entity_map[subject]
#         else:
#             resolved_subject = subject
#             
#         # Resolve object  
#         if obj in entity_map:
#             resolved_obj = entity_map[obj]
#         else:
#             resolved_obj = obj
#             
#         resolved_triples.append((resolved_subject, predicate, resolved_obj))
#     
#     all_triples = resolved_triples
#     print(f"✓ Applied coreference resolution")
#     
#     # ============ POST-PROCESSING ============
#     print(f"\nPost-processing {len(all_triples)} triples...")
#     
#     # Remove duplicates and filter
#     seen = set()
#     filtered_triples = []
#     
#     for subject, predicate, obj in all_triples:
#         # Create normalized version for deduplication
#         norm_triple = (subject.lower().strip(), predicate.lower().strip(), obj.lower().strip())
#         
#         if (norm_triple not in seen and 
#             len(subject) > 1 and len(obj) > 1 and
#             subject.lower() != obj.lower() and
#             predicate not in ['the', 'a', 'an']):
#             
#             seen.add(norm_triple)
#             filtered_triples.append((subject, predicate, obj))
#     
#     all_triples = filtered_triples
#     print(f"After deduplication: {len(all_triples)} triples")
#     
#     # Save results
#     if all_triples:
#         save_triples(all_triples, "extracted_triples.txt")
#         
#         print("\nEXTRACTED SCRUM KNOWLEDGE GRAPH:")
#         print("=" * 50)
#         for i, (subject, predicate, obj) in enumerate(all_triples[:20], 1):
#             print(f"  {i:2d}. ({subject}) --[{predicate}]--> ({obj})")
#         
#         if len(all_triples) > 20:
#             print(f"  ... and {len(all_triples) - 20} more triples")
#         
#         print(f"\nSuccess! Extracted {len(all_triples)} Scrum knowledge relationships")
#         
#         try:
#             plot_triples(all_triples[:20])
#         except Exception as e:
#             print(f"Visualization skipped: {e}")
#         
#         return all_triples
#     
#     else:
#         print("\nNo triples were successfully extracted")
#         return []


In [123]:
# ============ MAIN PIPELINE ============
# ============ ENHANCED CLEANING FUNCTIONS ============

def clean_entity(entity: str) -> str:
    if not entity:
        return ""
    entity = re.sub(r'\s+', ' ', entity.strip())
    entity = re.sub(r'^(The|A|An)\s+', '', entity, flags=re.IGNORECASE)
    entity = re.sub(r'[,;:.!?]+$', '', entity)
    entity = re.sub(r'\b(team|group|department)\b', lambda m: m.group().title(), entity, flags=re.IGNORECASE)
    return entity.title()


def clean_predicate(predicate: str) -> str:
    if not predicate:
        return "relates_to"
    predicate = predicate.lower().strip()
    predicate = re.sub(r'[^\w\s]', '', predicate)
    predicate = re.sub(r'\s+', '_', predicate)
    mappings = {
        'is_a': 'is', 'has_a': 'has', 'contain': 'contains', 'belong': 'belongs_to',
        'work': 'works_on', 'manage': 'manages', 'create': 'creates', 'use': 'uses',
        'participate': 'participates_in', 'attend': 'attends'
    }
    for k, v in mappings.items():
        if k in predicate:
            predicate = v
            break
    if predicate.endswith(('ing', 'ed')):
        predicate = predicate[:-3] if predicate.endswith('ing') else predicate[:-2]
    elif predicate.endswith('s') and len(predicate) > 3:
        predicate = predicate[:-1]
    return predicate


def is_valid_triple(subject: str, predicate: str, obj: str) -> bool:
    if len(subject) < 2 or len(obj) < 2: return False
    if subject.lower() == obj.lower(): return False
    bad_words = {'it', 'this', 'that', 'they', 'he', 'she', 'something', 'anything',
                 'everything', 'nothing', 'somewhere', 'anywhere', 'everyone', 'anyone'}
    if subject.lower() in bad_words or obj.lower() in bad_words: return False
    if predicate in ['is', 'has', 'does'] and len(predicate) < 4: return False
    return True


def save_triples(triples, filename="extracted_triples.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Knowledge Graph Triples\n# Total: {len(triples)} triples\n")
        f.write("# Format: (Subject) --[Predicate]--> (Object)\n\n")
        for i, (s, p, o) in enumerate(triples, 1):
            f.write(f"{i:3d}. ({s}) --[{p}]--> ({o})\n")
        f.write(f"\n# Extraction completed. {len(triples)} unique triples saved.\n")


# ============ SIMPLE FALLBACK RELATION EXTRACTOR ============

async def simple_relation_extraction(text_chunk, llm_model):    
    simple_prompt = f"""Extract relationships from this text as a simple JSON list:

    Text: {text_chunk}
    
    Return exactly this format:
    [
      {{"source": "entity1", "relation": "relationship", "target": "entity2"}},
      {{"source": "entity2", "relation": "relationship", "target": "entity3"}}
    ]
    
    Focus on: roles, responsibilities, artifacts, events, and their connections.
    Return only the JSON array, no other text.
    """
    
    try:
        response = await llm_model.ainvoke(simple_prompt)
        response_text = response.content if hasattr(response, 'content') else str(response)
        
        # Try to parse JSON
        try:
            relations = json.loads(response_text)
            if isinstance(relations, list):
                return relations
        except:
            pass
            
        return []
    except:
        return []


# ============ MAIN FUNCTION WITH ETA ============

async def main(input_source: str = "text.txt"):
    print("Starting Enhanced KG Extraction with Scrum Schema + Fallback Extractor\n")

    # Load text
    if os.path.exists(input_source):
        with open(input_source, "r", encoding="utf-8") as f:
            text = f.read().strip()
    else:
        text = input_source.strip()
    if len(text) < 20:
        print("Text too short")
        return []

    # Chunk text
    sentences = re.split(r'(?<=[.!?])\s+', text)
    texts, current_chunk = [], ""
    for sentence in sentences:
        if len(current_chunk + sentence) <= 800:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip(): texts.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip(): texts.append(current_chunk.strip())
    print(f"Text loaded: {len(text)} chars in {len(texts)} chunks")

    config = load_config()
    llm, embeddings = initialize_llm(config)
    print(f"LLM initialized: {config['llm']['model_name']}")

    all_triples = []

    # ----------------- DOCUMENT DISTILLER -----------------
    print("\nExtracting structured facts with Document Distiller...")
    distiller = DocumentsDistiller(llm_model=llm)
    IE_query = get_scrum_IE_query()
    all_facts = []

    start_time = time.time()

    for i, chunk in enumerate(texts):
        print(f"  Processing chunk {i+1}/{len(texts)}...")
    
        try:
            facts_result = await distiller.distill(
                documents=[chunk],
                IE_query=IE_query,
                output_data_structure=ScrumProjectFacts
            )
            
            if facts_result and hasattr(facts_result, 'facts'):
                chunk_facts = facts_result.facts
                all_facts.extend(chunk_facts)
                print(f"Extracted {len(chunk_facts)} facts")
            else:
                print(f"No facts extracted")
                
        except Exception as chunk_error:
            print(f"Error processing chunk {i+1}: {chunk_error}")
            continue
    
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (len(texts) - (i + 1))
        print(f"Elapsed: {elapsed:.1f}s | Avg/chunk: {avg_time:.1f}s | ETA remaining: {remaining/60:.1f} min")
    
    for fact in all_facts:
        s = clean_entity(fact.subject)
        o = clean_entity(fact.object)
        p = clean_predicate(fact.predicate)
        if is_valid_triple(s, p, o):
            all_triples.append((s, p, o))
    print(f"Converted to {len(all_triples)} valid triples")
    
        # ----------------- STAR WITH CORRECT ATTRIBUTES (llama3.1) -----------------
    print("\nExtracting with STAR...")
    
    use_fallback = False
    
    try:
        from itext2kg.logging_config import setup_logging, get_logger
        logger = get_logger(__name__)
        
        star_model = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)
        logger.info("STAR model initialized successfully")
    except Exception as e:
        print(f"STAR initialization failed: {e}")
        use_fallback = True
    
    start_time = time.time()
    
    for i, chunk in enumerate(texts):
        print(f"  Processing chunk {i + 1}/{len(texts)}...")
    
        if not use_fallback:
            try:
                kg_result = await star_model.build_graph(
                    sections=[chunk],
                    ent_threshold=0.6,      
                    rel_threshold=0.5,      
                    observation_date="2025-01-15"
                )
                
                if hasattr(kg_result, 'relationships') and kg_result.relationships:
                    chunk_triples = 0
                    
                    for relationship in kg_result.relationships:
                        try:
                            start_entity = getattr(relationship, 'startEntity', None)
                            end_entity = getattr(relationship, 'endEntity', None)
                            relation_name = getattr(relationship, 'name', 'relates_to')
                            
                            s = getattr(start_entity, 'name', 'unknown') if start_entity else 'unknown'
                            o = getattr(end_entity, 'name', 'unknown') if end_entity else 'unknown'
                            p = str(relation_name) if relation_name else 'relates_to'
    
                            s = clean_entity(str(s))
                            o = clean_entity(str(o))
                            p = clean_predicate(str(p))
    
                            if is_valid_triple(s, p, o):
                                all_triples.append((s, p, o))
                                chunk_triples += 1
                                
                        except Exception as rel_error:
                            continue
                    
                    print(f"STAR extracted {chunk_triples} triples")
                    
                else:
                    print(f"STAR found no relationships, using fallback...")
                    use_fallback = True
                    
            except Exception as chunk_error:
                print(f"STAR failed: {str(chunk_error)[:100]}... Using fallback...")
                use_fallback = True
    
        if use_fallback:
            try:
                relations = await simple_relation_extraction(chunk, llm)
                chunk_triples = 0
                for rel in relations:
                    if isinstance(rel, dict) and 'source' in rel and 'target' in rel and 'relation' in rel:
                        s = clean_entity(rel['source'])
                        o = clean_entity(rel['target'])
                        p = clean_predicate(rel['relation'])
                        
                        if is_valid_triple(s, p, o):
                            all_triples.append((s, p, o))
                            chunk_triples += 1
                
                print(f"Fallback extracted {chunk_triples} triples")
            except Exception as fallback_error:
                print(f"Both STAR and fallback failed: {fallback_error}")
                continue
    
        use_fallback = False
    
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (len(texts) - (i + 1))
        print(f"Elapsed: {elapsed:.1f}s | Avg/chunk: {avg_time:.1f}s | ETA remaining: {remaining / 60:.1f} min")
        
    # ----------------- POST-PROCESSING -----------------
    print("\nPost-processing triples...")
    seen, unique_triples = set(), []
    for triple in all_triples:
        if triple not in seen:
            seen.add(triple)
            unique_triples.append(triple)
    final_triples = unique_triples

    print(f"\nFinal Results: Total triples: {len(all_triples)}, Unique triples: {len(unique_triples)}")
    save_triples(final_triples)

    if final_triples:
        print(f"\nSample extracted triples:")
        for i, (s, p, o) in enumerate(final_triples[:5], 1):
            print(f"   {i}. ({s}) --[{p}]--> ({o})")
        if len(final_triples) > 5:
            print(f"   ... and {len(final_triples) - 5} more")

    return final_triples


In [124]:
#STAR with default Document distiler
async def mainDefaultWithoutCustomScheme(input_source: str = "text.txt"):
    print("Starting Enhanced Knowledge Graph Extraction with Document Distiller + iText2KG_Star\n")

    # ============ LOAD INPUT TEXTS ============

    texts = []
    try:
        if isinstance(input_source, str) and os.path.exists(input_source):
            print(f"Loading from file: {input_source}")
            with open(input_source, "r", encoding="utf-8") as f:
                content = f.read().strip()

                if len(content) > 2000:
                    sentences = content.split('. ')
                    current_chunk = ""
                    for sentence in sentences:
                        if len(current_chunk + sentence) <= 2000:
                            current_chunk += sentence + ". "
                        else:
                            if current_chunk:
                                texts.append(current_chunk.strip())
                            current_chunk = sentence + ". "
                    if current_chunk:
                        texts.append(current_chunk.strip())
                else:
                    texts = [content]

        elif isinstance(input_source, str) and len(input_source.strip()) > 10:
            print("Using direct text input")
            texts = [input_source.strip()]

        else:
            print(f"Invalid input source: {input_source}")
            return []

    except Exception as e:
        print(f"Error loading input: {e}")
        return []

    texts = [text for text in texts if len(text.strip()) > 20]

    if not texts:
        print("No valid texts found. Exiting.")
        return []

    print(f"Loaded {len(texts)} text segments")
    for i, text in enumerate(texts[:2]):
        print(f"   Text {i + 1} preview: {text[:100]}...")

    # ============ INITIALIZE LLM ============
    print("\nInitializing LLM and Embeddings...")
    config = load_config()
    llm, embeddings = initialize_llm(config)

    if not llm or not embeddings:
        print("Cannot proceed without LLM initialization")
        return []

    print(f"LLM initialized: {config['llm']['model_name']}")

    # ============ DOCUMENT DISTILLER ============
    print("\nExtracting structured facts with Document Distiller...")

    try:
        from itext2kg import DocumentsDistiller
        from itext2kg.models.schemas import Facts

        # Initialize Document Distiller
        document_distiller = DocumentsDistiller(llm_model=llm)

        IE_query = '''
            # DIRECTIVES:
            - Act like an experienced business analyst and information extractor.
            - You are analyzing text about software development, product management, and stakeholder relationships.
            - Extract clear, specific facts about:
              * People and their roles (e.g., Pat is a Product Owner)
              * Relationships between people and concepts
              * Processes and workflows
              * Systems and their purposes
              * Stakeholder needs and requirements
            - Focus on concrete, actionable facts rather than general statements.
            - Preserve specific names, roles, and relationships.
            - If you find examples (like flight booking), include them as specific facts.
            '''

        all_facts = []
        for i, text in enumerate(texts):
            print(f"Extracting facts from segment {i + 1}...")

            facts_result = await document_distiller.distill(
                documents=[text],
                IE_query=IE_query,
                output_data_structure=Facts
            )

            if hasattr(facts_result, 'facts') and facts_result.facts:
                all_facts.extend(facts_result.facts)
                print(f"  Extracted {len(facts_result.facts)} facts from segment {i + 1}")
                for j, fact in enumerate(facts_result.facts[:3]):
                    print(f"Fact {j + 1}: {fact[:100]}...")
            else:
                print(f"No facts extracted from segment {i + 1}")

        if not all_facts:
            print("Warning: No facts extracted, falling back to original text")
            all_facts = texts
        else:
            print(f"Total facts extracted: {len(all_facts)}")

    except Exception as e:
        print(f"Document Distiller failed: {e}")
        print("Falling back to original text segments")
        all_facts = texts

    # ============ KNOWLEDGE GRAPH CONSTRUCTION WITH iText2KG_Star ============
    print("\nBuilding Knowledge Graph with iText2KG_Star using extracted facts...")

    all_triples = []

    try:
        itext2kg_star = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)
        print("iText2KG_Star initialized successfully")

        print(f"Building knowledge graph from {len(all_facts)} facts/sections...")

        kg_result = await itext2kg_star.build_graph(
            sections=all_facts,
            ent_threshold=0.6,
            rel_threshold=0.6,
            max_tries=5,
            entity_name_weight=0.7,
            entity_label_weight=0.3,
            observation_date="2025-08-10"
        )

        print(f"Knowledge graph construction completed")
        print(f"Result type: {type(kg_result)}")

        # ============ EXTRACT TRIPLES FROM RESULT ============
        if hasattr(kg_result, 'relationships'):
            relationships = kg_result.relationships
            print(f"Found {len(relationships)} relationships")
        elif hasattr(kg_result, 'edges'):
            relationships = kg_result.edges
            print(f"Found {len(relationships)} edges")
        else:
            print("No relationships found in the result")
            relationships = []

        for i, rel in enumerate(relationships):
            try:
                if hasattr(rel, '__class__') and 'Relationship' in str(rel.__class__):
                    print(f"Processing Relationship object {i}")

                    subject = None
                    predicate = None
                    obj = None

                    if hasattr(rel, 'name'):
                        predicate = str(rel.name).strip()

                    if hasattr(rel, 'startEntity'):
                        start_entity = rel.startEntity
                        if hasattr(start_entity, 'name') and start_entity.name:
                            subject = str(start_entity.name).strip()
                        elif hasattr(start_entity, 'label') and start_entity.label:
                            subject = str(start_entity.label).strip()

                    if hasattr(rel, 'endEntity'):
                        end_entity = rel.endEntity
                        if hasattr(end_entity, 'name') and end_entity.name:
                            obj = str(end_entity.name).strip()
                        elif hasattr(end_entity, 'label') and end_entity.label:
                            obj = str(end_entity.label).strip()

                    if not predicate or len(predicate) < 2:
                        if subject and obj:
                            predicate = f"relates_to"
                            print(f"Warning: Empty predicate detected, using 'relates_to'")

                    if subject:
                        subject = subject.replace('_', ' ').title() if '_' in subject else subject
                    if obj:
                        obj = obj.replace('_', ' ').title() if '_' in obj else obj
                    if predicate:
                        predicate = predicate.replace('_', ' ').lower()

                    print(f"  Extracted - Subject: {subject}, Predicate: {predicate}, Object: {obj}")

                else:
                    print(f"Unknown relationship format at index {i}: {type(rel)}")
                    continue

                if subject and predicate and obj:
                    subject = str(subject).strip()
                    predicate = str(predicate).strip()
                    obj = str(obj).strip()

                    if (len(subject) > 1 and len(obj) > 1 and len(predicate) > 1 and
                            subject != obj and predicate not in ['is', 'has', 'the']):
                        all_triples.append((subject, predicate, obj))
                        print(f"Added triple: ({subject}) --[{predicate}]--> ({obj})")
                    else:
                        print(f"Skipped invalid triple: subject='{subject}', predicate='{predicate}', object='{obj}'")
                else:
                    print(f"Could not extract complete triple from relationship {i}")

            except Exception as e:
                print(f"Error processing relationship {i}: {e}")
                import traceback

                traceback.print_exc()
                continue

    except Exception as e:
        print(f"Knowledge graph construction failed: {e}")
        import traceback

        traceback.print_exc()
        return []

    # ============ POST-PROCESSING & OUTPUT ============
    print(f"\nProcessing Results...")

    if all_triples:
        original_count = len(all_triples)
        all_triples = list(set(all_triples))
        print(f"After deduplication: {len(all_triples)} triples (removed {original_count - len(all_triples)})")

        save_triples(all_triples, "extracted_triples.txt")

        print("\nEXTRACTED KNOWLEDGE GRAPH:")
        print("=" * 50)
        for i, (subject, predicate, obj) in enumerate(all_triples[:15], 1):
            print(f"  {i:2d}. ({subject}) --[{predicate}]--> ({obj})")

        if len(all_triples) > 15:
            print(f"  ... and {len(all_triples) - 15} more triples")

        print(f"\nSuccess! Extracted {len(all_triples)} knowledge relationships")

        try:
            plot_triples(all_triples[:20])
        except Exception as e:
            print(f"Visualization skipped: {e}")

        return all_triples

    else:
        print("\nNo triples were successfully extracted")
        return []

In [125]:
#10. Run the main() function

In [ ]:
if __name__ == "__main__":
    import nest_asyncio

    nest_asyncio.apply()


    async def run_star_pipeline():
        triples = await main("text.txt")
        print(f"\nPipeline completed! Found {len(triples)} triples")
        return triples


    loop = asyncio.get_event_loop()
    triples = loop.run_until_complete(run_star_pipeline())


🚀 Starting Enhanced KG Extraction with Scrum Schema + Fallback Extractor

📊 Text loaded: 9126 chars in 11 chunks
LLM and Embeddings initialized successfully (Ollama)
✅ LLM initialized: llama3.1

📋 Extracting structured facts with Document Distiller...
  Processing chunk 1/11...
    ✅ Extracted 3 facts
    ⏱️ Elapsed: 56.8s | Avg/chunk: 56.8s | ETA remaining: 9.5 min
  Processing chunk 2/11...
    ✅ Extracted 4 facts
    ⏱️ Elapsed: 101.1s | Avg/chunk: 50.5s | ETA remaining: 7.6 min
  Processing chunk 3/11...
    ✅ Extracted 2 facts
    ⏱️ Elapsed: 179.0s | Avg/chunk: 59.7s | ETA remaining: 8.0 min
  Processing chunk 4/11...
    ✅ Extracted 3 facts
    ⏱️ Elapsed: 219.0s | Avg/chunk: 54.8s | ETA remaining: 6.4 min
  Processing chunk 5/11...
    ✅ Extracted 2 facts
    ⏱️ Elapsed: 325.3s | Avg/chunk: 65.1s | ETA remaining: 6.5 min
  Processing chunk 6/11...
    ✅ Extracted 3 facts
    ⏱️ Elapsed: 383.6s | Avg/chunk: 63.9s | ETA remaining: 5.3 min
  Processing chunk 7/11...
